# 11 — Çok görevli model ve daha güçlü omurgalar (Adım 12–13), Kaggle

**Ayarlar:** Accelerator **GPU T4 x2** · Internet **On** · Secrets: `GH_TOKEN`. Sonra **Save Version → Save & Run All (Commit)**.

Yaklaşık **4 saat** sürer; tarayıcıyı kapatabilirsiniz. Kalori sonuçları ~1,5 saat sonra **ara kayıt** olarak GitHub'a gider,
sınıflandırma sonuçları en sonda. Bir eğitim hata verirse sonrakiler yine çalışır (`|| true`).

**Bu notebook'ta yeni olan:** kalori modellerinin son katmanı artık train ortalamasını tahmin ederek başlıyor
(`init_regression_head`). Notebook 10'daki EfficientNet-B0 bu yüzden 104 kcal'de kalmıştı; ResNet-50 ve EfficientNet-B0
kalori modelleri tablo tutarlı olsun diye yeniden eğitiliyor (eskiler `*_timminit` adıyla duruyor).

In [ ]:
# 1) KURULUM — her oturumda ilk bu hücre çalışır
import os, subprocess
from kaggle_secrets import UserSecretsClient

REPO = "https://github.com/traldo44/kalori-tahmini.git"
GIT_NAME, GIT_EMAIL = "Mert Aydıngüneş", "aydingunesmert@gmail.com"   # ← başkası çalıştırırsa değiştirsin
OWNER = "mert"                                          # ← results.csv'deki owner sütunu

os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GH_TOKEN")
# Token'ı hiçbir dosyaya yazmadan git'e verir; yalnızca bu oturumun hafızasında durur.
subprocess.run(["git", "config", "--global", "credential.helper",
                '!f() { echo username=x-access-token; echo "password=$GH_TOKEN"; }; f'], check=True)

os.chdir("/kaggle/working")
if os.path.isdir("kalori-tahmini"):
    subprocess.run(["git", "-C", "kalori-tahmini", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/kaggle/working/kalori-tahmini")
subprocess.run(["git", "config", "user.name", GIT_NAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)

os.environ["KALORI_DATA"] = "/tmp/kalori-data"   # veri çıktı klasörüne (/kaggle/working) girmesin
os.environ["KALORI_OWNER"] = OWNER
print("Repo hazır →", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2) VERİ — Nutrition5k (~1,4 GB) + Food-101 25 sınıflık alt küme. Aynı oturumda tekrar çalışırsa atlar.
!python scripts/download_nutrition5k.py --out /tmp/kalori-data/nutrition5k
!python scripts/get_food101.py fetch --out /tmp/kalori-data

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU yok → Settings → Accelerator → GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0))

## Kalori (Nutrition5k) — Tablo 2 ve 3
Çıta: notebook 10'daki ResNet-50, test MAE 59,7 kcal.

In [ ]:
# Adım 11 tekrar: düzeltilmiş baş başlatmasıyla (~15 dk + ~15 dk)
!python -m src.train_reg --backbone efficientnet_b0 || true
!python -m src.train_reg --backbone resnet50 || true

In [ ]:
# Adım 12: çok görevli — kalori + kütle + yağ + karbonhidrat + protein (~15 dk)
!python -m src.train_reg --backbone resnet50 --targets calories mass fat carb protein || true

In [ ]:
# Adım 13: daha güçlü omurgalar (~20 dk + ~40 dk)
!python -m src.train_reg --backbone convnext_tiny || true
!python -m src.train_reg --backbone vit_base_patch16_224 --name vit_b16_reg || true

### Ara kayıt — kalori sonuçları GitHub'a

In [ ]:
!git add results && git commit -q -m "Adım 11-13: Kaggle kalori sonuçları (init düzeltmesi, çok görevli, ConvNeXt-T, ViT-B/16)" && git pull --rebase -q && git push -q && echo "GitHub'a gönderildi ✓"
# yüklenen model dosyası silinir → son hücre aynı dosyayı tekrar yüklemez
!for f in checkpoints/*_reg*.pt; do [ -e "$f" ] && python scripts/share_checkpoint.py upload "$f" && rm "$f"; done

## Sınıflandırma (Food-101, 25 sınıf) — Tablo 1
Çıta: EfficientNet-B0 test F1 0,917 / ResNet-50 0,912.

In [ ]:
# Adım 13 (~35 dk + ~80 dk)
!python -m src.train_cls --backbone convnext_tiny || true
!python -m src.train_cls --backbone vit_base_patch16_224 --name vit_b16_cls || true

## Sonuçları gönder

In [ ]:
!git add results && git commit -q -m "Adım 13: Kaggle sınıflandırma sonuçları (ConvNeXt-T, ViT-B/16)" && git pull --rebase -q && git push -q && echo "GitHub'a gönderildi ✓"
# yüklenen model dosyası silinir → son hücre aynı dosyayı tekrar yüklemez
!for f in checkpoints/*.pt; do [ -e "$f" ] && python scripts/share_checkpoint.py upload "$f" && rm "$f"; done